# Published scores documented in clinical notes (regex, exploratory)

Feasibility test: how often do clinicians write the published prognostic scores (or their
missing inputs) directly in the chart? `build_published_scores` computes the scores from
structured data only; this notebook checks whether progress notes could fill the gaps:

- **Scores as documented**: IPI (incl. R-IPI / NCCN-IPI), MELD (incl. MELD-Na / MELD 3.0),
  ALBI (grade or score), CAPRA (incl. CAPRA-S), mGPS / GPS, LIPI, RMH, IMDC (Heng), MSKCC (Motzer).
- **Performance status**: ECOG and KPS, the performance-status item of IPI/IMDC/MSKCC. The
  latest in-window value (KPS converted to ECOG) completes the builder's full calculated scores.

For each score's eligible population (from `published_scores_df.csv.gz`, treatment anchor),
it reports the share of patients with a mention in a note dated 0-`LOOKBACK_DAYS` days before
the anchor (the same no-leakage rule as the lab window), any time before the anchor, and any
time at all. It also compares note values with the structured scores where both exist.

Regex output is noisy. Read the snippet samples before trusting a coverage number. Known traps:
`ipi` is also shorthand for ipilimumab (dose mentions like `ipi 1 mg/kg` are dropped; coverage is
only counted inside the aggressive-NHL population), `MSKCC` is also the hospital and `Heng` a
surname (risk-group patterns require the group word next to the name), and ECOG ranges like
`ECOG 1-2` keep both ends (`value`, `value_hi`).

The patterns and score mapping live in `pipelines.preprocessing.extract_note_scores`, whose
`published_scores_note_df*.csv.gz` output is the `regex` score source in
`figures.prep.published_scores` and the ECOG input of `build_published_scores` (both run
from `1_data/01b_published_scores.ipynb`). Pattern unit
tests are in `tests/test_extract_note_scores.py`. This notebook is the manual review of that
extraction: it writes nothing the pipeline reads. Optional output (`SAVE_OUTPUTS`):
`FEATURE_PATH/published_scores_note_mentions.csv.gz` (one row per match, with a text
snippet, so it contains PHI and stays on the cluster) and
`FEATURE_PATH/published_scores_note_coverage.csv.gz`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import polars as pl


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from config import FEATURE_PATH
from pipelines.preprocessing import extract_note_scores as ens
from shared.published_scores import CATALOG_SCORE_IDS

ANCHOR = "treatment"
LOOKBACK_DAYS = ens.DEFAULT_LOOKBACK_DAYS
NOTE_TYPES = ens.DEFAULT_NOTE_TYPES  # add "PATHOLOGY_NOTES" / "IMAGING_NOTES" if wanted
SAVE_OUTPUTS = True
N_SNIPPETS = 8

pl.Config.set_tbl_rows(60)
pl.Config.set_fmt_str_lengths(200)
print(f"repo root: {REPO_ROOT}")

## Patterns

`ens.MENTION_PATTERNS` holds one entry per (mention, measure, regex). Every regex has a named
`value` group; `variant` (e.g. `R-IPI` vs `IPI`), `value_hi` (upper end of an ECOG range) and
`unit` (used only to drop ipilimumab doses) are optional. Matching is case-insensitive, on the
Rust regex engine (no look-arounds; exclusions are applied after extraction). `ens.VALID_RANGE`
drops values outside a score's possible range (e.g. `MELD 3` is not a MELD). Edit patterns in
the module and add a case to `tests/test_extract_note_scores.py` for every failure the snippet
review below turns up.

In [ ]:
pl.DataFrame(
    [(m, measure, pattern) for m, measure, pattern in ens.MENTION_PATTERNS],
    schema=["mention", "measure", "pattern"], orient="row",
)

## Cohort, anchor and eligibility

Eligibility comes from the builder's treatment-anchor output, so the populations match the
structured-score analysis exactly. If that file is missing, every cohort patient counts as
eligible for everything (coverage numbers are then cohort-wide).

In [ ]:
cohort = ens.load_anchor_cohort(ANCHOR)

SCORE_IDS = CATALOG_SCORE_IDS
scores_path = os.path.join(FEATURE_PATH, "published_scores_df.csv.gz")
if os.path.exists(scores_path):
    structured = pl.read_csv(scores_path, schema_overrides={"DFCI_MRN": pl.Int64}, infer_schema_length=None)
else:
    print(f"{scores_path} not found; treating every cohort patient as eligible")
    structured = cohort.select("DFCI_MRN", *[pl.lit(True).alias(f"{s}__eligible") for s in SCORE_IDS])

# Which structured-score populations each note mention is counted against.
MENTION_POPULATIONS = {
    "mgps": ["mgps"], "rmh": ["rmh"], "lipi": ["lipi"], "albi": ["albi"], "meld": ["meld"],
    "capra": ["capra_mod"], "ipi": ["ipi"], "imdc": ["imdc"], "mskcc": ["mskcc"],
    "ecog": ["ipi", "imdc", "mskcc", "all"],
    "kps": ["ipi", "imdc", "mskcc", "all"],
}
populations = {"all": cohort.select("DFCI_MRN")}
for score_id in SCORE_IDS:
    populations[score_id] = structured.filter(pl.col(f"{score_id}__eligible").fill_null(False)).select("DFCI_MRN")
print(pl.DataFrame({"population": list(populations), "n": [p.height for p in populations.values()]}))

## Scan notes

`ens.scan_note_mentions` is the same scan the pipeline stage runs, here with `snippets=True`:
lazy and streaming, filtered to cohort patients and a cheap keyword prefilter before any
extraction, with `EVENT_DATE` localized to a naive date as in `build_cohort`.

In [ ]:
mentions = ens.scan_note_mentions(cohort, NOTE_TYPES, snippets=True)
print(f"{mentions.height:,} parsed mentions")
mentions.group_by("mention", "measure").len().sort("mention", "measure")

## Coverage by population

Share of each population with at least one parsed mention, in three windows:
`0-LOOKBACK_DAYS` days before the anchor (no leakage, usable as a score input),
any time before the anchor, and any time (upper bound on documentation).

In [ ]:
windows = {
    f"le{LOOKBACK_DAYS}d": pl.col("days_before_anchor").is_between(0, LOOKBACK_DAYS),
    "before_anchor": pl.col("days_before_anchor") >= 0,
    "any_time": pl.lit(True),
}
rows = []
for mention, pops in MENTION_POPULATIONS.items():
    mention_rows = mentions.filter(pl.col("mention") == mention)
    for pop in pops:
        pop_ids = populations[pop]
        row = {"mention": mention, "population": pop, "n_population": pop_ids.height}
        for label, window in windows.items():
            n = mention_rows.filter(window).join(pop_ids, on="DFCI_MRN", how="semi")["DFCI_MRN"].n_unique()
            row[f"n_{label}"] = n
            row[f"pct_{label}"] = round(100 * n / pop_ids.height, 1) if pop_ids.height else None
        rows.append(row)
coverage = pl.DataFrame(rows)
coverage

## Values and snippets for manual review

Value distribution per mention within the no-leakage window and population, then a random
sample of snippets per (mention, measure). Check the snippets before trusting coverage.

In [ ]:
in_window = windows[f"le{LOOKBACK_DAYS}d"]
for mention, pops in MENTION_POPULATIONS.items():
    rows_ = mentions.filter((pl.col("mention") == mention) & in_window).join(populations[pops[0]], on="DFCI_MRN", how="semi")
    if rows_.is_empty():
        print(f"--- {mention} ({pops[0]}): no mentions in window")
        continue
    print(f"--- {mention} ({pops[0]})")
    print(rows_.group_by("measure", "variant", "value").len().sort("measure", "len", descending=[False, True]).head(15))

In [ ]:
for (mention, measure), group in mentions.group_by("mention", "measure", maintain_order=True):
    sample = group.sample(n=min(N_SNIPPETS, group.height), seed=0)
    print(f"\n=== {mention} / {measure}: {group.height:,} matches ===")
    for snippet, value in sample.select("snippet", "value").iter_rows():
        print(f"  [{value}] ...{snippet}...")

## Agreement with the structured scores

For each patient, the latest in-window mention (per mention and measure) against the
builder's structured value. Structured IPI/IMDC/MSKCC are the full scores, whose performance
status comes from this same note extraction, so they exist only for patients with a documented
ECOG/KPS. All scores should match up to lab-timing differences; a note value above the
structured points can also mean a documented item the structured data missed.

In [ ]:
STRUCTURED_COLUMN = {
    "mgps": "mgps__points", "rmh": "rmh__points", "lipi": "lipi__points", "albi": "albi__continuous",
    "meld": "meld__continuous", "capra": "capra_mod__points", "ipi": "ipi__points",
    "imdc": "imdc__points", "mskcc": "mskcc__points",
}
latest = (
    mentions.filter(in_window)
    .sort("days_before_anchor")
    .group_by("DFCI_MRN", "mention", "measure", maintain_order=True)
    .first()
)
for mention, column in STRUCTURED_COLUMN.items():
    group_col = column.split("__")[0] + "__group"
    if column not in structured.columns:
        continue
    joined = latest.filter(pl.col("mention") == mention).join(
        structured.select(
            "DFCI_MRN",
            pl.col(column).cast(pl.Float64, strict=False).alias("structured"),
            pl.col(group_col).cast(pl.String).alias("structured_group"),
        ),
        on="DFCI_MRN", how="inner",
    ).filter(pl.col("structured").is_not_null())
    if joined.is_empty():
        print(f"--- {mention}: no patients with both a note value and a structured value")
        continue
    print(f"--- {mention}: {joined.height} patients with both")
    numeric = joined.filter(pl.col("value_num").is_not_null() & (pl.col("measure") != "grade"))
    if not numeric.is_empty():
        print(numeric.select(
            pl.len().alias("n"),
            (pl.col("value_num") == pl.col("structured").round(0)).mean().alias("frac_equal"),
            (pl.col("value_num") >= pl.col("structured").round(0)).mean().alias("frac_note_ge_structured"),
            pl.corr("value_num", "structured", method="spearman").alias("spearman"),
        ))
    print(joined.group_by("measure", "value", "structured_group").len().sort("measure", "value", "structured_group"))

## Scores as the prep evaluates them

`ens.note_score_frame` maps the in-window mentions onto the catalog score scale (latest note,
ties take the highest value), exactly as written by the pipeline stage. Counts are within each
score's eligible population, which is the regex source's cohort before the text-score and
outcome joins in `figures.prep.published_scores`. The last block is `ecog__note_value`, the
performance status the builder uses for the full calculated IPI/IMDC/MSKCC (the `_noecog`
variants have no note source).

In [ ]:
note_frame = ens.note_score_frame(mentions, cohort, LOOKBACK_DAYS)
for score_id in ens.NOTE_SCORES:
    documented = (
        note_frame.join(populations[score_id], on="DFCI_MRN", how="semi")
        .filter(pl.col(f"{score_id}__note_value").is_not_null())
    )
    print(f"--- {score_id}: {documented.height:,} of {populations[score_id].height:,} eligible documented")
    if not documented.is_empty():
        print(documented.group_by(f"{score_id}__note_group").len().sort(f"{score_id}__note_group"))
for score_id in ("ipi", "imdc", "mskcc"):
    documented = (
        note_frame.join(populations[score_id], on="DFCI_MRN", how="semi")
        .filter(pl.col("ecog__note_value").is_not_null())
    )
    print(f"--- ECOG for {score_id}: {documented.height:,} of {populations[score_id].height:,} eligible documented")
    if not documented.is_empty():
        print(documented.group_by("ecog__note_source", "ecog__note_value").len().sort("ecog__note_source", "ecog__note_value"))

## Save

In [ ]:
if SAVE_OUTPUTS:
    mentions_path = os.path.join(FEATURE_PATH, "published_scores_note_mentions.csv.gz")
    coverage_path = os.path.join(FEATURE_PATH, "published_scores_note_coverage.csv.gz")
    mentions.write_csv(mentions_path, compression="gzip")
    coverage.write_csv(coverage_path, compression="gzip")
    print(f"wrote {mentions_path}\nwrote {coverage_path}")